# 14 Mailbox: package model output for collaborators

A **mailbox** is one folder in your scratch space where finished products go for other people:

```
/scratch/ttg2015/mailbox/
    <recipient>/
        <YYYY-MM-DD>_<label>/          one delivery
            <experiment>_rv<r_v>_3d.nc   one netCDF per run
            README.txt                 what is in it, units, grid, how it was made
            MANIFEST.json              files, sizes, checksums, source runs, git commit
```

Each delivery is written once and never edited. A new export goes in a new dated folder, so the
collaborator can always tell what changed. This notebook starts with **3D fields as netCDF**. The
functions are written so they can move into a `samheat/mailbox.py` script later.

Run the cells in order: **levers → plan (size check, writes nothing) → write → (optional) share**.

In [ ]:
# RUN THIS CELL FIRST: lets this notebook find the samheat code in this repo (no install needed)
import os, sys, pathlib
def _find_repo():
    starts = [pathlib.Path(os.environ.get("JPY_SESSION_NAME", ".")).resolve(), pathlib.Path.cwd().resolve()]
    for s in starts:
        for d in [s, *s.parents]:
            if (d / "samheat" / "__init__.py").exists():
                return d
    for d in [pathlib.Path("/scratch/ttg2015/Torch/ExtremeHeat/SAM_HEAT"), pathlib.Path.home() / "SAM_HEAT"]:
        if (d / "samheat" / "__init__.py").exists():
            return d
    raise FileNotFoundError("Could not find the SAM_HEAT folder. Set REPO by hand, e.g. REPO = pathlib.Path('/scratch/.../SAM_HEAT')")
REPO = _find_repo()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
import samheat
print("using samheat from", pathlib.Path(samheat.__file__).parent)

## 1. Levers

In [ ]:
from pathlib import Path

MAILBOX    = Path('/scratch/ttg2015/mailbox')   # the mailbox root
RECIPIENT  = 'collaborator'                     # one subfolder per person, e.g. their surname
LABEL      = 'vddgrid_3d_preview'               # short name for this delivery

EXPERIMENT = 'vdd2025_vddgrid'                  # under SITE.work_root, or a full path
RV_LIST    = [0, 2000]                          # one file per run (stage2_rv<r_v>)
RUN_NAME   = 'stage2_rv{rv:g}'

FIELDS     = ['U', 'V', 'W', 'TABS', 'QV', 'QN', 'QP']   # None = every field in the files
TMIN, TMAX = 55.0, 60.0                         # days (Stage 2 output is days 30-60, every 3 h)
EVERY      = 1                                  # keep every Nth snapshot in the window
ZMAX       = 16000.                             # m; None = full column (above 16 km is the sponge)
COMPLEVEL  = 4                                  # netCDF zlib level 0-9 (4 is a good trade-off)
CONTACT    = 'Tyler Greiner'                    # written into the files and README
NOTE       = 'Preview of the 3D fields. Snapshot output, not time averages.'

## 2. Functions (to become `samheat/mailbox.py`)

In [ ]:
import datetime, hashlib, json, os, subprocess
import numpy as np
from samheat.paths import SITE
from samheat.io.sam3d import open_run_3d
from samheat.analysis.explore import experiment_dir


def delivery_dir(mailbox=MAILBOX, recipient=RECIPIENT, label=LABEL, date=None):
    date = date or datetime.date.today().isoformat()
    return Path(mailbox) / recipient / f'{date}_{label}'


def run_dir(experiment, rv, run_name=RUN_NAME):
    return experiment_dir(experiment) / run_name.format(rv=rv)


def open_selection(rdir, fields=FIELDS, tmin=TMIN, tmax=TMAX, every=EVERY, zmax=ZMAX):
    # Lazy: only file headers are read here; data are read while writing.
    ds = open_run_3d(rdir, fields=fields, tmin=tmin, tmax=tmax, zmax=zmax, lazy=True)
    return ds.isel(time=slice(None, None, every))


def describe(ds, experiment, rv, rdir):
    # CF-style metadata so the file explains itself in ncdump or Panoply.
    ds = ds.copy()
    ds['time'].attrs.update(units='day', long_name='model time in days since the start of the run')
    ds['x'].attrs.update(long_name='x distance (periodic domain)')
    ds['y'].attrs.update(long_name='y distance (periodic domain)')
    ds.attrs = dict(
        title=f'SAM 6.11 RCE, {Path(experiment).name}, r_v = {rv:g} s/m: 3D snapshot fields',
        model='System for Atmospheric Modeling (SAM) 6.11, FWRCE configuration',
        experiment=Path(experiment).name, r_v_s_per_m=float(rv), source_run=str(rdir),
        grid=f'{ds.sizes["x"]} x {ds.sizes["y"]} points, dx = {float(ds.x[1] - ds.x[0]):g} m, '
             f'{ds.sizes["z"]} levels up to {float(ds.z[-1]):.0f} m',
        units_note='U V W m/s; TABS K; QV QN QP g/kg (as written by SAM); PP Pa',
        contact=CONTACT, note=NOTE,
        created=datetime.datetime.now().isoformat(timespec='seconds'))
    return ds


def encoding(ds, complevel=COMPLEVEL):
    # One snapshot per chunk: a reader can pull a single time without decompressing the rest.
    return {v: dict(zlib=complevel > 0, complevel=complevel, dtype='float32',
                    chunksizes=(1,) + ds[v].shape[1:]) for v in ds.data_vars}


def estimate_gb(ds):
    return sum(v.size * 4 for v in ds.data_vars.values()) / 1e9   # uncompressed float32


def sha256(path, block=1 << 24):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(block), b''):
            h.update(b)
    return h.hexdigest()


def git_commit():
    try:
        return subprocess.run(['git', '-C', str(REPO), 'rev-parse', '--short', 'HEAD'],
                              capture_output=True, text=True).stdout.strip()
    except Exception:
        return 'unknown'

## 3. Plan: what would be written (reads headers only, writes nothing)

In [ ]:
out_dir = delivery_dir()
plan = []
for rv in RV_LIST:
    rdir = run_dir(EXPERIMENT, rv)
    try:
        ds = open_selection(rdir)
    except (FileNotFoundError, ValueError, KeyError) as e:
        print(f'r_v={rv:g}: SKIPPED -- {e}'); continue
    name = f'{Path(EXPERIMENT).name}_rv{rv:g}_3d.nc'
    plan.append(dict(rv=rv, rdir=rdir, ds=ds, name=name))
    t = ds.time.values
    print(f'r_v={rv:g}: {ds.sizes["time"]} snapshots (days {t[0]:.2f}-{t[-1]:.2f}), '
          f'{ds.sizes["z"]} levels, fields {list(ds.data_vars)}, ~{estimate_gb(ds):.2f} GB uncompressed')
total = sum(estimate_gb(p['ds']) for p in plan)
print(f'\nDelivery folder: {out_dir}')
print(f'Total ~{total:.2f} GB before compression (QN/QP compress very well, U/V/W/TABS less so).')
if out_dir.exists():
    print('NOTE: this delivery folder already exists. Change LABEL, or delete it yourself if you mean to redo it.')

## 4. Write the delivery

Writes one file per run, then the README and MANIFEST. It refuses to overwrite an existing delivery.

In [ ]:
if out_dir.exists():
    raise FileExistsError(f'{out_dir} exists; deliveries are write-once. Change LABEL.')
out_dir.mkdir(parents=True)
records = []
for p in plan:
    path = out_dir / p['name']
    ds = describe(p['ds'], EXPERIMENT, p['rv'], p['rdir'])
    print(f"writing {path.name} ...", flush=True)
    tmp = path.with_suffix('.nc.part')
    ds.to_netcdf(tmp, encoding=encoding(ds))
    tmp.rename(path)                          # a half-written file never carries the final name
    records.append(dict(file=path.name, bytes=path.stat().st_size, sha256=sha256(path),
                        r_v=float(p['rv']), source_run=str(p['rdir']),
                        fields=list(ds.data_vars), snapshots=int(ds.sizes['time']),
                        days=[float(ds.time[0]), float(ds.time[-1])],
                        levels=int(ds.sizes['z']), z_top_m=float(ds.z[-1])))
    print(f"  {records[-1]['bytes'] / 1e9:.2f} GB")

manifest = dict(delivery=out_dir.name, recipient=RECIPIENT, created=datetime.datetime.now().isoformat(timespec='seconds'),
                experiment=str(EXPERIMENT), samheat_commit=git_commit(), window_days=[TMIN, TMAX], every=EVERY,
                zmax_m=ZMAX, files=records)
(out_dir / 'MANIFEST.json').write_text(json.dumps(manifest, indent=2))

lines = [f'{LABEL} -- prepared by {CONTACT}, {manifest["created"][:10]}', '', NOTE, '',
         'Model: SAM 6.11 (System for Atmospheric Modeling), radiative-convective equilibrium,',
         'doubly periodic 128 km x 128 km domain, 1 km grid. "r_v" is the surface evaporative',
         'resistance (s/m): 0 = ocean-like, larger = drier surface.', '',
         'Files (one per run; dims time, z, y, x):']
lines += [f'  {r["file"]}  r_v={r["r_v"]:g}  {r["snapshots"]} snapshots, days {r["days"][0]:.2f}-{r["days"][1]:.2f}, '
          f'{r["levels"]} levels to {r["z_top_m"]:.0f} m, {r["bytes"] / 1e9:.2f} GB' for r in records]
lines += ['', 'Variables: U, V, W (m/s); TABS absolute temperature (K); QV water vapour, QN cloud',
          'condensate, QP precipitating water (g/kg). Coordinates: time (days), z (m), p(z) (hPa), x, y (m).',
          'Values are instantaneous snapshots every 3 model hours.', '',
          'Open with: xarray.open_dataset(file) in Python, or ncdump -h file for a summary.',
          'Checksums (sha256) are in MANIFEST.json.']
(out_dir / 'README.txt').write_text('\n'.join(lines) + '\n')
print('\n'.join(lines))
print(f'\nDone: {out_dir}')

## 5. (Optional) let the collaborator read it

Only if they have an account on Torch. This makes **this delivery** readable by everyone on the system. It
also lets them pass through the folders above it, which list nothing unless they know the path. Run it
deliberately. If you'd rather share with one person only, ask Torch support about ACLs (`setfacl`) instead.

In [ ]:
SHARE = False   # set True and re-run to apply

if SHARE:
    for d in [MAILBOX.parent, MAILBOX, out_dir.parent]:
        os.chmod(d, os.stat(d).st_mode | 0o001)                 # o+x: pass through, no listing
    os.chmod(out_dir, 0o755)
    for f in out_dir.iterdir():
        os.chmod(f, 0o644)
    print('shared (read-only):', out_dir)
else:
    print('Not shared. Collaborator without Torch access: download the folder (scp/rsync or OnDemand) and send it on.')